<div align="center">

# NNDL Final Project: Flight Delay Forecasting

**Kevin Brugnera · Sara Pasquato · Libero Pollini**

IDs: 2196578 · (inserite) · 2206131

</div>

---

First and foremost, we explore the 2022 chain dataset.

### Imports

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Subset
import torch.optim as optim
from torch.utils.data import DataLoader

import pickle
import copy
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [2]:

# from google.colab import files # opens interactive window to choose file(s) to upload files
# import os

# uploaded = files.upload()

# for name in uploaded.keys():
#     print("Uploaded:", name)


In [3]:
# define column names as global variables
DATE_COLS=["FL_DATE"]
DATETIME_COLS=["CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME",
               "ARR_TIME", "WHEELS_OFF", "WHEELS_ON", ]
TIMEDELTA_MINS_COLS=["DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "TAXI_IN", "CRS_ELAPSED_TIME",
                     "ACTUAL_ELAPSED_TIME", "AIR_TIME",	]
INT_COLS=["OP_CARRIER_FL_NUM", "FLIGHTS", "MONTH", "DAY_OF_MONTH",
          "DAY_OF_WEEK", "ORIGIN_INDEX", "DEST_INDEX"]
STR_COLS=["OP_CARRIER", "ORIGIN", "DEST"]
FLOAT_COLS=["O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD", "O_LATITUDE",
             "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE"]
size_set_check=set(DATE_COLS+DATETIME_COLS+TIMEDELTA_MINS_COLS+INT_COLS+STR_COLS+FLOAT_COLS)
print(f"Total individual features (should be 34): {len(size_set_check)}")

Total individual features (should be 34): 34


In [4]:
# to reload in case data_loader.py has been changed in the meantime:
# import importlib
# import data_loader
# importlib.reload(data_loader) 
from data_loader import download_dataset, load_dataset_pytorch

/home/libero/nndl_final_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load dataset

In [5]:
year = 2022


In [6]:
file_path = download_dataset(
    year, year + 1, mode="sequential"
)  # or "sequential" for pre-made chains)


Path data/chain/2022/train_flight_chain_2022.pt already exists! Skipping it
Path data/chain/2022/val_flight_chain_2022.pt already exists! Skipping it
Path data/chain/2022/test_flight_chain_2022.pt already exists! Skipping it


In [7]:
# load chain datasets with pytorch
# split_types = ["train", "val", "test"] 

# loaded_data = {}

# for split in split_types:
#     file_path = f"data/chain/{year}/{split}_flight_chain_{year}.pt"
#     loaded_data[split] = torch.load(file_path, weights_only=False)

#     print(f"--- File: (split: {split}, year: {year}) ---")
#     print("Data Type:", type(loaded_data[split]))

loaded_data=load_dataset_pytorch(year=2022)

--- Read file: (split: train, year: 2022) ---
--- Read file: (split: val, year: 2022) ---
--- Read file: (split: test, year: 2022) ---


In [8]:
# --- 1. Compute mean/std of valid (non-padded) train delays, once ---
train_dataset = loaded_data["train"]
all_delays = train_dataset.tensors[4]      # [N, seq_len, 2]
all_valid_lens = train_dataset.tensors[3]  # [N]

seq_len = all_delays.shape[1]
mask = torch.arange(seq_len).unsqueeze(0) < all_valid_lens.unsqueeze(1)  # [N, seq_len]

valid_delays = all_delays[mask].float()  # [total_valid_steps, 2] -- only as big as valid entries, no dataset copy

delay_mean = valid_delays.mean(dim=0)  # shape [2]
delay_std = valid_delays.std(dim=0)    # shape [2]

print(f"Delay mean (ARR, DEP): {delay_mean}")
print(f"Delay std (ARR, DEP): {delay_std}")

# --- 2. Scale/unscale helpers (operate on tensors on-the-fly, no dataset copy) ---
def scale_targets(targets, mean=delay_mean, std=delay_std):
    return (targets - mean.to(targets.device)) / std.to(targets.device)

def unscale_targets(scaled, mean=delay_mean, std=delay_std):
    return scaled * std.to(scaled.device) + mean.to(scaled.device)

Delay mean (ARR, DEP): tensor([ 6.8322, 12.3763])
Delay std (ARR, DEP): tensor([54.0997, 52.0273])


Samples are fewer than those reported in [Aeolus](https://arxiv.org/pdf/2510.26616) tabular dataset because they filtered to retain only coherent flight chains operated by the same aircraft (see Table 8, page 17).

In [9]:
N_train = len(loaded_data["train"])
N_val = len(loaded_data["val"])
N_test = len(loaded_data["test"])

N_chains = N_train + N_val + N_test

print(f"Total number of flight chains: {N_chains}")
print(f"Train samples: {N_train} ({N_train*100/N_chains:.2f}%)")
print(f"Validation samples: {N_val} ({N_val*100/N_chains:.2f}%)")
print(f"Test samples: {N_test} ({N_test*100/N_chains:.2f}%)")


Total number of flight chains: 5192960
Train samples: 2985432 (57.49%)
Validation samples: 1126506 (21.69%)
Test samples: 1081022 (20.82%)


Example of a flight chain sample

In [10]:
# 1. Extract the first sample from the training dataset
sample = loaded_data["train"][0]

print(f"Sample Type: {type(sample)}")
print(f"Total components in the sample tuple: {len(sample)}\n")
print("-" * 60)

# 2. Unpack each component of the 5-element tuple into properly named variables
dense_feat, sparse_feat, labels, valid_lens, delays = sample

# 3. Print details and relative values for each component with descriptive labels based on source code
print("1. Dense Features (Continuous / Meteorological Features):")
print(f"   - Shape: {dense_feat.shape} (Sequence Length x 7)")
print(f"   - Cols: ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD', 'FLIGHTS']")
print(f"    - Values:\n{dense_feat}\n")

print("2. Sparse Features (Categorical / Temporal Features):")
print(f"   - Shape: {sparse_feat.shape} (Sequence Length x 8)")
print(f"   - Cols: ['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX', 'OP_CARRIER', 'OP_CARRIER_FL_NUM']")
print(f"    - Values:\n{sparse_feat}\n")

print("3. Binary Labels (Flight Delay Indicators > 15 mins):")
print(f"   - Shape: {labels.shape} (Sequence Length x 2)")
print(f"   - [(ARR_DELAY > 15), (DEP_DELAY > 15)]")
print(f"   - Values:\n{labels}\n")

print("4. Valid Sequence Lengths (Metadata):")
print(f"   - Description: Effective number of valid flights in the chain before padding")
print(f"   - Shape: {valid_lens.shape}")
print(f"   - Values: {valid_lens}\n")

print("5. Raw Delays (Ground Truth):")
print(f"   - Shape: {delays.shape} (Sequence Length x 2)")
print(f"   - Cols: ['ARR_DELAY', 'DEP_DELAY']")
print(f"   - Values:\n{delays}")
print("-" * 60)

Sample Type: <class 'tuple'>
Total components in the sample tuple: 5

------------------------------------------------------------
1. Dense Features (Continuous / Meteorological Features):
   - Shape: torch.Size([6, 7]) (Sequence Length x 7)
   - Cols: ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD', 'FLIGHTS']
    - Values:
tensor([[ 6.7000, 19.4000,  0.0000,  0.0000,  9.4000, 14.8000,  1.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])

2. Sparse Features (Categorical / Temporal Features):
   - Shape: torch.Size([6, 8]) (Sequence Length x 8)
   - Cols: ['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX', 'OP_CA

In [11]:
print(valid_lens)

tensor(1)


### LSTM model

#### Metrics

In [12]:
import torch
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


def compute_all_metrics(pred_delays, true_delays, true_labels, valid_lens):
    """
    Compute regression (MSE, MAE) and classification (Accuracy, Precision, Recall, F1, AUC)
    for arrival and departure delays, masking out padded time steps.

    Args:
        pred_delays: Tensor [batch, seq_len, 2] - predicted delays (arr, dep)
        true_delays: Tensor [batch, seq_len, 2] - ground truth delays (arr, dep)
        true_labels: Tensor [batch, seq_len, 2] - binary labels (1 if delay > 15 min)
        valid_lens:  Tensor [batch] - number of valid (non-padded) steps per sequence

    Returns:
        dict: nested dict with metrics for arrival ('arr') and departure ('dep'),
              each containing regression and classification metrics.
    """
    batch_size, seq_len = pred_delays.shape[0], pred_delays.shape[1]
    device = pred_delays.device

    # Mask: True for valid positions, False for padding
    mask = torch.arange(seq_len, device=device).expand(
        batch_size, seq_len
    ) < valid_lens.unsqueeze(1)  # [batch, seq_len]

    # Flatten only valid entries for each target column
    pred_arr = pred_delays[:, :, 0][mask].flatten().cpu().numpy()
    pred_dep = pred_delays[:, :, 1][mask].flatten().cpu().numpy()
    true_arr = true_delays[:, :, 0][mask].flatten().cpu().numpy()
    true_dep = true_delays[:, :, 1][mask].flatten().cpu().numpy()
    label_arr = true_labels[:, :, 0][mask].flatten().cpu().numpy()
    label_dep = true_labels[:, :, 1][mask].flatten().cpu().numpy()

    # Helper for classification metrics
    def clf_metrics(y_true, y_pred, y_score):
        """
        Compute accuracy, precision, recall, F1 from binary predictions,
        and AUC from raw scores.
        """
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_score)  # raw scores for AUC
        except ValueError:
            auc = 0.5  # fallback if only one class present
        return acc, prec, rec, f1, auc

    # Regression metrics
    mse_arr = mean_squared_error(true_arr, pred_arr)
    mae_arr = mean_absolute_error(true_arr, pred_arr)
    mse_dep = mean_squared_error(true_dep, pred_dep)
    mae_dep = mean_absolute_error(true_dep, pred_dep)

    # Classification: threshold at 15 minutes
    pred_bin_arr = (pred_arr > 15).astype(int)
    pred_bin_dep = (pred_dep > 15).astype(int)

    acc_arr, prec_arr, rec_arr, f1_arr, auc_arr = clf_metrics(
        label_arr, pred_bin_arr, pred_arr
    )
    acc_dep, prec_dep, rec_dep, f1_dep, auc_dep = clf_metrics(
        label_dep, pred_bin_dep, pred_dep
    )

    metrics = {
        "arr_reg": {"MSE": mse_arr, "MAE": mae_arr},
        "dep_reg": {"MSE": mse_dep, "MAE": mae_dep},
        "arr_clf": {
            "Accuracy": acc_arr,
            "Precision": prec_arr,
            "Recall": rec_arr,
            "F1": f1_arr,
            "AUC": auc_arr,
        },
        "dep_clf": {
            "Accuracy": acc_dep,
            "Precision": prec_dep,
            "Recall": rec_dep,
            "F1": f1_dep,
            "AUC": auc_dep,
        },
    }
    return metrics

#### Model definition

Fist LSTM model on fixed length chains (6 flights)

In [13]:
# # Filtering

# # 1. Define the exact target length required
# target_length = 5

# # 2. Initialize a dictionary to store the filtered datasets
# filtered_loaded_data = {}

# # 3. Iterate through each split ('train', 'val', 'test') in the loaded dataset
# for split_name, dataset in loaded_data.items():
#     print(f"Processing split: {split_name} (Original samples: {len(dataset)})")
    
#     # Filter samples where the sequence length (dense_feat shape[0]) equals target_length
#     filtered_samples = [
#         sample for sample in dataset 
#         if sample[0].shape[0] == target_length
#     ]
    
#     print(f"-> Filtered samples (length == {target_length}): {len(filtered_samples)}")
    


In [14]:
#     # If samples match the condition, reconstruct into a TensorDataset
#     if len(filtered_samples) > 0:
#         dense_list = torch.stack([s[0] for s in filtered_samples])
#         sparse_list = torch.stack([s[1] for s in filtered_samples])
#         labels_list = torch.stack([s[2] for s in filtered_samples])
#         valid_lens_list = torch.stack([s[3] for s in filtered_samples])
#         delays_list = torch.stack([s[4] for s in filtered_samples])
        
#         filtered_loaded_data[split_name] = TensorDataset(
#             dense_list, sparse_list, labels_list, valid_lens_list, delays_list
#         )
#     else:
#         # Keep an empty or None placeholder if no samples match
#         filtered_loaded_data[split_name] = None

# print("\nFiltering complete. All splits are stored in 'filtered_loaded_data'.")

LSTM with recurrent units + linear regression layer

In [15]:
class FlightChainLSTM(nn.Module):
    def __init__(self, dense_input_dim, sparse_input_dim, hidden_dim=6, output_dim=2):
        """
        Params:
        - dense_input_dim: Number of continuous features (e.g., 7 meteorological/dense features).
        - sparse_input_dim: Number of categorical/sparse features (e.g., 8 discrete attributes).
        - hidden_dim: Number of hidden units in the LSTM layer 
        - output_dim: Number of regression outputs (e.g., 2 for arrival and departure delays).
        """
        super(FlightChainLSTM, self).__init__()
        
        # Total input dimension combining dense and sparse features per time step
        self.total_input_dim = dense_input_dim + sparse_input_dim
        
        self.hidden_dim = hidden_dim
        
        # 1. LSTM Layer configured with 6 hidden units and batch_first=True
        # Input shape expected: [batch_size, sequence_length (6), total_input_dim]
        self.lstm = nn.LSTM(
            input_size=self.total_input_dim,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True
        )
        
        # 2. Fully Connected Regression Layer to map LSTM outputs to the target delay values
        self.regressor = nn.Linear(self.hidden_dim, output_dim)
        
    def forward(self, dense_feat, sparse_feat):
        """
        Params:
        - dense_feat: Tensor of shape [batch_size, 6, 7]
        - sparse_feat: Tensor of shape [batch_size, 6, 8] (converted to float for concatenation)
        """
        x = torch.cat((dense_feat, sparse_feat.float()), dim=2)
        lstm_out, (hn, cn) = self.lstm(x)
        predictions = self.regressor(lstm_out)
        
        return predictions

#### Hyperparameters & Configuration

In [21]:
DENSE_DIM = 7  # O_TEMP, D_TEMP, O_PRCP, D_PRCP, O_WSPD, D_WSPD, FLIGHTS
SPARSE_DIM = 8 # MONTH, DAY_OF_WEEK, CRS_ARR_TIME_HOUR, etc.
HIDDEN_UNITS = 32 # original 32
OUTPUT_REGRESSION_DIM = 2 # ARR_DELAY and DEP_DELAY

BATCH_SIZE = 64 # original 64
LEARNING_RATE = 0.01 # original 0.001
NUM_EPOCHS = 50 # original 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# 2. Create DataLoaders from your filtered splits - NO: use original (already padded/truncated) dataset
# train_loader = DataLoader(filtered_loaded_data['train'], batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(filtered_loaded_data['val'], batch_size=BATCH_SIZE, shuffle=False)

fraction = 0.1  # for trial runs ---> about 500_000 samples

train_indices = torch.randperm(len(loaded_data["train"]))[
    : int(fraction * len(loaded_data["train"]))
]
val_indices = torch.randperm(len(loaded_data["val"]))[
    : int(fraction * len(loaded_data["val"]))
]
test_indices = torch.randperm(len(loaded_data["test"]))[
    : int(fraction * len(loaded_data["test"]))
]

train_loader = DataLoader(
    Subset(loaded_data["train"], train_indices),
    batch_size=BATCH_SIZE,
    shuffle=True,  # cant hurt in training
    num_workers=4,
    pin_memory=True,  # for faster but more memory consuming training
)
val_loader = DataLoader(
    Subset(loaded_data["val"], val_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,  # bc True unnecessary, would add computation overhead and lose reproducibility
    num_workers=4,
    pin_memory=True,
)
test_loader = DataLoader(
    Subset(loaded_data["test"], test_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

Using device: cpu


In [22]:
# 3. Instantiate the Model, Loss, and Optimizer
model = FlightChainLSTM(
    dense_input_dim=DENSE_DIM,
    sparse_input_dim=SPARSE_DIM,
    hidden_dim=HIDDEN_UNITS,
    output_dim=OUTPUT_REGRESSION_DIM,
).to(DEVICE)

# MSE Loss is standard for regression tasks (predicting continuous delay values)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

#### Training

In [23]:
best_val_loss = float("inf")
patience_counter = 0
early_stop = False
best_model_state = None

train_loss_history = []
val_loss_history = []
# 4. Training Loop
for epoch in range(NUM_EPOCHS):
    print(f"Doing epoch {epoch}...")
    model.train()
    running_train_loss = 0.0

    for batch in train_loader:
        dense_feat, sparse_feat, labels, valid_lens, targets = [
            tensor.to(DEVICE) for tensor in batch
        ]

        # Zero the gradients from the previous step
        optimizer.zero_grad()

        # Forward pass
        predictions = model(dense_feat, sparse_feat)

        # loss
        loss = criterion(predictions, scale_targets(targets.float()))

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_train_loss += loss.item() * dense_feat.size(0)

    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    # 5. Validation Loop
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            dense_feat, sparse_feat, labels, valid_lens, targets = [
                tensor.to(DEVICE) for tensor in batch
            ]

            predictions = model(dense_feat, sparse_feat)
            loss = criterion(predictions, scale_targets(targets.float()))

            running_val_loss += loss.item() * dense_feat.size(0)

    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    
    train_loss_history.append(epoch_train_loss)
    val_loss_history.append(epoch_val_loss)
    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}"
    )

    
    # Early stopping (if val MSE doesn't improve for 5 epochs)
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f"Early stopping triggered at epoch {epoch+1}")
            early_stop = True
            break

# After loop, load best (according to all validation history) model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("Training and evaluation process completed successfully.")

Doing epoch 0...


/home/libero/nndl_final_project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch [1/50] | Train Loss: 0.2053 | Val Loss: 0.2189
Doing epoch 1...
Epoch [2/50] | Train Loss: 0.2051 | Val Loss: 0.2180
Doing epoch 2...
Epoch [3/50] | Train Loss: 0.2051 | Val Loss: 0.2183
Doing epoch 3...
Epoch [4/50] | Train Loss: 0.2051 | Val Loss: 0.2178
Doing epoch 4...
Epoch [5/50] | Train Loss: 0.2051 | Val Loss: 0.2176
Doing epoch 5...
Epoch [6/50] | Train Loss: 0.2052 | Val Loss: 0.2183
Doing epoch 6...
Epoch [7/50] | Train Loss: 0.2051 | Val Loss: 0.2177
Doing epoch 7...


KeyboardInterrupt: 

#### Testing / evaluation

In [ ]:
model.eval()
all_preds, all_targets, all_labels, all_lens = [], [], [], []

with torch.no_grad():
    for batch in test_loader:
        dense_feat, sparse_feat, labels, valid_lens, targets = [
            t.to(DEVICE) for t in batch
        ]
        preds = model(dense_feat, sparse_feat)
        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())
        all_labels.append(labels.cpu())
        all_lens.append(valid_lens.cpu())  # flight chain lengths, directly from dataset

test_preds = torch.cat(all_preds, dim=0)
test_preds = unscale_targets(test_preds)  # back to real minutes
test_targets = torch.cat(all_targets, dim=0)  # already unscaled ground truth, unchanged
test_labels = torch.cat(all_labels, dim=0)
test_lens = torch.cat(all_lens, dim=0)

test_metrics = compute_all_metrics(test_preds, test_targets, test_labels, test_lens)

print("\n=== Test Set Metrics ===")
for target in ["arr", "dep"]:
    print(f"\n{target.upper()} Regression:")
    for k, v in test_metrics[f"{target}_reg"].items():
        print(f"  {k}: {v:.4f}")
    print(f"{target.upper()} Classification:")
    for k, v in test_metrics[f"{target}_clf"].items():
        print(f"  {k}: {v:.4f}")


=== Test Set Metrics ===

ARR Regression:
  MSE: 2561.6479
  MAE: 25.2060
ARR Classification:
  Accuracy: 0.8007
  Precision: 0.0000
  Recall: 0.0000
  F1: 0.0000
  AUC: 0.4857

DEP Regression:
  MSE: 2419.8433
  MAE: 23.0841
DEP Classification:
  Accuracy: 0.7969
  Precision: 1.0000
  Recall: 0.0004
  F1: 0.0007
  AUC: 0.4836


In [ ]:
# save metrics history and best model and dump them to a pickle file

epochs_run = (
    epoch + 1
)  # actual number of epochs completed (accounts for early stopping)

results = {
    "train_loss_history": train_loss_history,
    "val_loss_history": val_loss_history,
    "best_val_loss": best_val_loss,
    #'best_model_state': best_model_state,
    "test_metrics": test_metrics,
    "delay_mean": delay_mean,
    "delay_std": delay_std,
    # might be heavy, but if wanna test later:
    #'test_preds': test_preds,
    #'test_targets': test_targets,
    #'test_labels': test_labels,
    #'test_lens': test_lens,
    "epochs_run": epochs_run,
    "hyperparams": {
        "hidden_units": HIDDEN_UNITS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "fraction": fraction,
    },
}

with open("training_results_ten_percent.pkl", "wb") as f:
    pickle.dump(results, f)

# Save best model separately
torch.save(best_model_state, "best_model_ten_percent.pth")